In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [ ]:
token_df

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import VideoAutoencoder, SequenceAutoencoder

In [ ]:
def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

def visualize(clean, recon, name="Reconstruction"):
    import matplotlib.pyplot as plt

    try:
        # Get both channels, frame 0 of first batch element
        clean_mri= clean[0, 0, 0].cpu().numpy()
    except RuntimeError:
        clean_mri = clean[0, 0, 0].detach().cpu().numpy()

    recon_mri = recon[0, 0, 0].detach().cpu().numpy()

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_mri, cmap='gray')
    plt.title("Original (MRI)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(recon_mri, cmap='gray')
    plt.title(f"{name} (MRI)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(f"video_visualization/beus/{name.lower()}.png", dpi=300)
    plt.close("all")

def combined_loss(y_true, y_pred, warped_x, criterion, plot=False, ratio=1):
    diff_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    diff_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]
    loss = criterion(diff_true, diff_pred) * (1 - ratio) + criterion(y_true, y_pred) * ratio

    if plot:
        visualize(diff_true, diff_pred, name="Reconstruction (differential)")
        visualize(y_true, y_pred, name="Reconstruction")
        visualize(y_true, warped_x, name="Transformation")

    return loss

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}

art_model = VideoAutoencoder(
    input_shape=(10, 56, 112),
    input_channels=2,
    latent_dim=1024,
    use_mlp=True,
    stn_mode="separate",
    grid_sample_cpu_fallback=True,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_beus_articulatory_features(art_f, start, end)

    if articulatory_features is None:
        continue

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    try:
        art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    except RuntimeError:
        continue
    batch.clear()

    # ===== Articulatory =====
    art_out, art_warped_x, art_z = art_model(art_batch, return_warped_x=True, return_latent=True)
    art_loss = combined_loss(art_batch, art_out, art_warped_x, art_criterion, plot=batch_idx % 30 == 0, ratio=0.5)

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/beus/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/beus/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/beus/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/beus/art_model_training_log.csv", index=False)

In [ ]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

In [ ]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/beus/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/beus/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/beus/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/beus/ac_model_training_log.csv", index=False)